In [2]:
# Ensure PyTorch is installed (run this first)
import sys, subprocess, importlib

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

try:
    import torch
    print(f'PyTorch is available, version {torch.__version__}')
except Exception:
    print('PyTorch not found. Installing...')
    # Install PyTorch CPU version (if you need GPU, use the official PyTorch command)
    install('torch')
    importlib.invalidate_caches()
    import torch
    print(f'Installed PyTorch, version {torch.__version__}')

# If ModuleNotFoundError persists, restart the kernel and re-run cells.

PyTorch not found. Installing...
Installed PyTorch, version 2.9.1+cpu
Installed PyTorch, version 2.9.1+cpu


## thư viện cần thiết

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import os

# (Tùy chọn) Kiểm tra xem có GPU không
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

Đang sử dụng thiết bị: cpu


## Task 1: Tải và Tiền xử lý Dữ liệu

In [ ]:
# --- 1. Viết hàm đọc file .conllu ---

def load_conllu(file_path):
    """
    Đọc file CoNLL-U và trả về danh sách các câu.
    Mỗi câu là một danh sách các cặp (word, upos_tag).
    """
    all_sentences = []
    current_sentence = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                # Nếu gặp dòng trống và câu hiện tại có dữ liệu
                if current_sentence:
                    all_sentences.append(current_sentence)
                    current_sentence = [] # Bắt đầu câu mới
            elif line.startswith('#'):
                # Bỏ qua các dòng comment
                continue
            else:
                # Phân tích dòng dữ liệu
                parts = line.split('\t')
                # Bỏ qua các multi-word token (ví dụ: '1-2 ...')
                if '-' in parts[0]: 
                    continue
                
                word = parts[1]  # Cột 2 (FORM)
                tag = parts[3]   # Cột 4 (UPOS)
                current_sentence.append((word, tag))
                
    # Thêm câu cuối cùng nếu file không kết thúc bằng dòng trống
    if current_sentence:
        all_sentences.append(current_sentence)
        
    return all_sentences

# --- Tải dữ liệu ---
train_file = "../../data/UD_English-EWT/en_ewt-ud-train.conllu"
dev_file = "../../data/UD_English-EWT/en_ewt-ud-dev.conllu"
try:
    train_data = load_conllu(train_file)
    dev_data = load_conllu(dev_file)
    
    print(f"Đã tải thành công {len(train_data)} câu train và {len(dev_data)} câu dev.")
    print("\n--- Ví dụ câu đầu tiên (train): ---")
    print(train_data[0])

except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file dữ liệu. Hãy chắc chắn bạn chạy code từ thư mục gốc")
    print(f"Và file tồn tại ở: {train_file}")


# --- 2. Xây dựng Từ điển (Vocabulary) ---

# Thêm 2 token đặc biệt: <PAD> (cho padding) và <UNK> (cho từ không biết)
# Đặt <PAD> = 0 vì đây là giá trị mặc định cho padding_idx
word_to_ix = {"<PAD>": 0, "<UNK>": 1}
tag_to_ix = {"<PAD>": 0} # Nhãn <PAD> sẽ có ignore_index = 0

# Chỉ xây dựng từ điển từ dữ liệu train
for sentence in train_data:
    for word, tag in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)
        if tag not in tag_to_ix:
            tag_to_ix[tag] = len(tag_to_ix)

# Tạo các từ điển tra cứu ngược (hữu ích cho việc debug và Task 5)
ix_to_word = {i: w for w, i in word_to_ix.items()}
ix_to_tag = {i: t for t, i in tag_to_ix.items()}


print("\n--- Xây dựng Từ điển ---")
print(f"Kích thước từ điển từ (word_to_ix): {len(word_to_ix)}")
print(f"Kích thước từ điển nhãn (tag_to_ix): {len(tag_to_ix)}")

# Lưu giá trị PAD_INDEX để dùng sau
PAD_INDEX = tag_to_ix["<PAD>"]

Đã tải thành công 12544 câu train và 2001 câu dev.

--- Ví dụ câu đầu tiên (train): ---
[('Al', 'PROPN'), ('-', 'PUNCT'), ('Zaman', 'PROPN'), (':', 'PUNCT'), ('American', 'ADJ'), ('forces', 'NOUN'), ('killed', 'VERB'), ('Shaikh', 'PROPN'), ('Abdullah', 'PROPN'), ('al', 'PROPN'), ('-', 'PUNCT'), ('Ani', 'PROPN'), (',', 'PUNCT'), ('the', 'DET'), ('preacher', 'NOUN'), ('at', 'ADP'), ('the', 'DET'), ('mosque', 'NOUN'), ('in', 'ADP'), ('the', 'DET'), ('town', 'NOUN'), ('of', 'ADP'), ('Qaim', 'PROPN'), (',', 'PUNCT'), ('near', 'ADP'), ('the', 'DET'), ('Syrian', 'ADJ'), ('border', 'NOUN'), ('.', 'PUNCT')]

--- Xây dựng Từ điển ---
Kích thước từ điển từ (word_to_ix): 19675
Kích thước từ điển nhãn (tag_to_ix): 18


## Task 2 - Tạo PyTorch Dataset và DataLoader

In [7]:
# --- 1. Tạo lớp POSDataset ---
class POSDataset(Dataset):
    def __init__(self, data, word_to_ix, tag_to_ix):
        self.data = data
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix
        self.unk_index = word_to_ix["<UNK>"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Lấy câu tại chỉ số idx
        sentence = self.data[idx]
        
        # Chuyển từ và nhãn sang chỉ số (index)
        # Sử dụng .get(word, self.unk_index) để xử lý các từ không có trong từ điển
        sentence_indices = [self.word_to_ix.get(word, self.unk_index) for word, tag in sentence]
        tag_indices = [self.tag_to_ix[tag] for word, tag in sentence]
        
        # Trả về tensor
        return torch.tensor(sentence_indices), torch.tensor(tag_indices)

# --- 2. Tạo hàm collate_fn và DataLoader ---

def collate_fn(batch):
    """
    Hàm này nhận vào một list các (sentence_tensor, tag_tensor)
    và pad chúng để có cùng độ dài.
    """
    # 1. Tách sentences và tags ra
    # batch là list: [(sent1, tag1), (sent2, tag2), ...]
    sentences, tags = zip(*batch)
    
    # 2. Pad sentences
    # sentences_padded có shape (batch_size, max_len)
    sentences_padded = pad_sequence(
        sentences, 
        batch_first=True, 
        padding_value=word_to_ix["<PAD>"]
    )
    
    # 3. Pad tags
    # tags_padded có shape (batch_size, max_len)
    tags_padded = pad_sequence(
        tags, 
        batch_first=True, 
        padding_value=tag_to_ix["<PAD>"]
    )
    
    return sentences_padded, tags_padded

# --- Khởi tạo Dataset và DataLoader ---
BATCH_SIZE = 32

train_dataset = POSDataset(train_data, word_to_ix, tag_to_ix)
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn
)

dev_dataset = POSDataset(dev_data, word_to_ix, tag_to_ix)
dev_loader = DataLoader(
    dev_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn
)

# --- Kiểm tra một batch ---
print("--- Kiểm tra DataLoader ---")
s_batch, t_batch = next(iter(train_loader))
print(f"Shape của batch câu: {s_batch.shape}")
print(f"Shape của batch nhãn: {t_batch.shape}")
print(f"Câu đầu tiên (đã pad): \n{s_batch[0]}")
print(f"Nhãn đầu tiên (đã pad): \n{t_batch[0]}")

--- Kiểm tra DataLoader ---
Shape của batch câu: torch.Size([32, 33])
Shape của batch nhãn: torch.Size([32, 33])
Câu đầu tiên (đã pad): 
tensor([4527,   92,  130, 6787,   25,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0])
Nhãn đầu tiên (đã pad): 
tensor([ 9,  8, 13,  3,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])


## Task 3 - Xây dựng Mô hình RNN

In [8]:
class SimpleRNNForTokenClassification(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, tagset_size, padding_idx):
        super(SimpleRNNForTokenClassification, self).__init__()
        
        # 1. Lớp Embedding
        # padding_idx=padding_idx nói với nn.Embedding bỏ qua index 0 khi học
        self.embedding = nn.Embedding(
            vocab_size, 
            embedding_dim, 
            padding_idx=padding_idx
        )
        
        # 2. Lớp RNN
        # batch_first=True vì DataLoader của chúng ta có shape (batch_size, seq_len)
        self.rnn = nn.RNN(
            embedding_dim, 
            hidden_dim, 
            batch_first=True
        )
        
        # 3. Lớp Linear (Fully Connected)
        # Ánh xạ từ hidden_dim (output của RNN) sang tagset_size (số lượng nhãn)
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentences):
        # Input 'sentences' shape: (batch_size, seq_len)
        
        # 1. Qua lớp Embedding
        # embeds shape: (batch_size, seq_len, embedding_dim)
        embeds = self.embedding(sentences)
        
        # 2. Qua lớp RNN
        # rnn_out shape: (batch_size, seq_len, hidden_dim)
        # hidden shape: (1, batch_size, hidden_dim) - chúng ta không cần dùng hidden state
        rnn_out, _ = self.rnn(embeds)
        
        # 3. Qua lớp Linear
        # logits shape: (batch_size, seq_len, tagset_size)
        logits = self.fc(rnn_out)
        
        return logits

## Task 4 - Khởi tạo Huấn luyện 

In [9]:
# --- 1. Khởi tạo ---

# Hyperparameters
VOCAB_SIZE = len(word_to_ix)
TAGSET_SIZE = len(tag_to_ix)
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
EPOCHS = 5 # Bạn có thể tăng lên 10-20 để có kết quả tốt hơn

# Khởi tạo mô hình
model = SimpleRNNForTokenClassification(
    VOCAB_SIZE, 
    EMBEDDING_DIM, 
    HIDDEN_DIM, 
    TAGSET_SIZE, 
    padding_idx=PAD_INDEX
).to(device)

# Khởi tạo Loss function
# ignore_index=PAD_INDEX (tức là 0)
# bảo CrossEntropyLoss bỏ qua các vị trí padding khi tính loss
loss_function = nn.CrossEntropyLoss(ignore_index=PAD_INDEX)

# Khởi tạo Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("--- Khởi tạo Mô hình ---")
print(model)

--- Khởi tạo Mô hình ---
SimpleRNNForTokenClassification(
  (embedding): Embedding(19675, 100, padding_idx=0)
  (rnn): RNN(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=18, bias=True)
)


## Task 5 (Phần 1) - Viết hàm Đánh giá

In [10]:
def evaluate(loader):
    """Tính accuracy trên tập dữ liệu (train hoặc dev)."""
    model.eval() # Chuyển mô hình sang chế độ đánh giá (tắt dropout, v.v.)
    
    total_correct = 0
    total_tokens = 0
    
    with torch.no_grad(): # Tắt việc tính gradient
        for sentences, tags in loader:
            # Chuyển dữ liệu sang device (GPU/CPU)
            sentences = sentences.to(device)
            tags = tags.to(device)
            
            # Forward pass
            # logits shape: (batch_size, seq_len, tagset_size)
            logits = model(sentences)
            
            # Lấy dự đoán (chỉ số có giá trị lớn nhất)
            # predictions shape: (batch_size, seq_len)
            predictions = torch.argmax(logits, dim=2)
            
            # --- Chỉ tính accuracy trên các token không phải padding ---
            # 1. Tạo mặt nạ (mask)
            # mask shape: (batch_size, seq_len)
            mask = (tags != PAD_INDEX)
            
            # 2. So sánh dự đoán và nhãn thật TẠI các vị trí mask=True
            correct = (predictions == tags)[mask].sum().item()
            num_tokens = mask.sum().item()
            
            total_correct += correct
            total_tokens += num_tokens
            
    return total_correct / total_tokens

## Task 4 & 5 (Phần 2) - Vòng lặp Huấn luyện & Báo cáo

In [11]:
# --- 2. Viết vòng lặp huấn luyện ---

print("--- Bắt đầu Huấn luyện ---")

for epoch in range(EPOCHS):
    
    # --- Training ---
    model.train() # Chuyển mô hình sang chế độ huấn luyện
    running_loss = 0.0
    
    for i, (sentences, tags) in enumerate(train_loader):
        # Chuyển dữ liệu sang device (GPU/CPU)
        sentences = sentences.to(device)
        tags = tags.to(device)
        
        # 1. Xóa gradient cũ
        optimizer.zero_grad()
        
        # 2. Forward pass
        # logits shape: (batch_size, seq_len, tagset_size)
        logits = model(sentences)
        
        # 3. Tính loss
        # CrossEntropyLoss yêu cầu input shape: (N, C) và target shape: (N)
        # Vì vậy, chúng ta cần reshape logits và tags
        loss = loss_function(
            logits.view(-1, TAGSET_SIZE), # Shape: (batch*seq_len, tagset_size)
            tags.view(-1)                  # Shape: (batch*seq_len)
        )
        
        # 4. Backward pass (tính đạo hàm)
        loss.backward()
        
        # 5. Cập nhật trọng số
        optimizer.step()
        
        running_loss += loss.item()
        
        if (i + 1) % 100 == 0: # In loss sau mỗi 100 batch
            print(f"  Epoch {epoch+1}, Batch {i+1}/{len(train_loader)}, Train Loss: {loss.item():.4f}")
            
    avg_train_loss = running_loss / len(train_loader)

    # --- Evaluating ---
    train_acc = evaluate(train_loader)
    dev_acc = evaluate(dev_loader)
    
    print("-" * 50)
    print(f"KẾT THÚC EPOCH {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Train Acc:  {train_acc:.4f}")
    print(f"  Dev Acc:    {dev_acc:.4f}")
    print("-" * 50)

print("--- Hoàn tất Huấn luyện! ---")

--- Bắt đầu Huấn luyện ---
  Epoch 1, Batch 100/392, Train Loss: 1.1113
  Epoch 1, Batch 100/392, Train Loss: 1.1113
  Epoch 1, Batch 200/392, Train Loss: 0.9693
  Epoch 1, Batch 200/392, Train Loss: 0.9693
  Epoch 1, Batch 300/392, Train Loss: 0.8272
  Epoch 1, Batch 300/392, Train Loss: 0.8272
--------------------------------------------------
KẾT THÚC EPOCH 1/5
  Train Loss: 1.1309
  Train Acc:  0.7687
  Dev Acc:    0.7418
--------------------------------------------------
--------------------------------------------------
KẾT THÚC EPOCH 1/5
  Train Loss: 1.1309
  Train Acc:  0.7687
  Dev Acc:    0.7418
--------------------------------------------------
  Epoch 2, Batch 100/392, Train Loss: 0.6260
  Epoch 2, Batch 100/392, Train Loss: 0.6260
  Epoch 2, Batch 200/392, Train Loss: 0.7316
  Epoch 2, Batch 200/392, Train Loss: 0.7316
  Epoch 2, Batch 300/392, Train Loss: 0.5897
  Epoch 2, Batch 300/392, Train Loss: 0.5897
--------------------------------------------------
KẾT THÚC EPOCH

### Dự đoán (Nâng cao)

In [12]:
def predict_sentence(sentence_str):
    """Nhận vào một chuỗi (str) và dự đoán nhãn POS."""
    model.eval()
    
    # 1. Tách từ
    tokens = sentence_str.split()
    
    # 2. Chuyển từ sang chỉ số
    indices = [word_to_ix.get(w, word_to_ix["<UNK>"]) for w in tokens]
    
    # 3. Chuyển sang tensor và thêm chiều batch (batch_size = 1)
    # shape: (1, seq_len)
    sentence_tensor = torch.tensor(indices).unsqueeze(0).to(device)
    
    # 4. Dự đoán
    with torch.no_grad():
        # logits shape: (1, seq_len, tagset_size)
        logits = model(sentence_tensor)
        
        # predictions shape: (1, seq_len)
        predictions = torch.argmax(logits, dim=2)
    
    # 5. Chuyển chỉ số dự đoán (tensor) về nhãn (string)
    # .squeeze(0) để bỏ chiều batch, .tolist() để chuyển về list Python
    pred_indices = predictions.squeeze(0).tolist()
    pred_tags = [ix_to_tag[i] for i in pred_indices]
    
    # 6. Trả về kết quả
    return list(zip(tokens, pred_tags))

# --- Thử dự đoán ---
print("\n--- Thử dự đoán ---")

test_sent_1 = "From the AP comes this story"
print(f"Câu: '{test_sent_1}'")
print(f"Dự đoán: {predict_sentence(test_sent_1)}\n")

test_sent_2 = "This model is pretty good"
print(f"Câu: '{test_sent_2}'")
print(f"Dự đoán: {predict_sentence(test_sent_2)}\n")

test_sent_3 = "The old man is running"
print(f"Câu: '{test_sent_3}'")
print(f"Dự đoán: {predict_sentence(test_sent_3)}\n")


--- Thử dự đoán ---
Câu: 'From the AP comes this story'
Dự đoán: [('From', 'ADP'), ('the', 'DET'), ('AP', 'NOUN'), ('comes', 'VERB'), ('this', 'DET'), ('story', 'NOUN')]

Câu: 'This model is pretty good'
Dự đoán: [('This', 'PRON'), ('model', 'NOUN'), ('is', 'AUX'), ('pretty', 'ADV'), ('good', 'ADJ')]

Câu: 'The old man is running'
Dự đoán: [('The', 'DET'), ('old', 'ADJ'), ('man', 'NOUN'), ('is', 'AUX'), ('running', 'VERB')]

